# Aula 03 - Pandas, Numpy e bibliotecas gráficas

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**11/08/2026 - Sprint 1 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula03.ipynb)

## O que este notebook é

É a exploração de dados (EDA) das **cinco séries** do case, agora com Pandas de verdade. A Aula 01
leu um CSV com `csv.reader`, na mão. A Aula 02 repetiu isso para cinco arquivos, ainda na mão, para
medir os seis pilares de qualidade. Hoje o `pandas.DataFrame` substitui as duas leituras: os
mesmos números saem em uma fração do código.

## Ao final deste notebook você terá

1. carregado as cinco séries em `DataFrame`, com `pandas.read_csv`;
2. rodado `describe()` e comparado com a checagem de sanidade de `dados/README.md`;
3. calculado a variação percentual trimestre a trimestre com `numpy.diff` e encontrado o maior
   salto de cada série;
4. plotado duas das cinco séries com Matplotlib e Seaborn, procurando sazonalidade e outliers;
5. confirmado, com `isna()`, que as cinco séries não têm valor ausente, e por quê.


## 1. Onde estão os arquivos

Mesma resolução de caminho das Aulas 01 e 02, agora para as cinco séries: funciona no repositório
clonado (onde os CSVs estão em `../dados/`) e no Colab (onde são baixados da versão publicada do
repositório).


In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]

BASE_LOCAL = os.path.join("..", "dados")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            urllib.request.urlretrieve(BASE_BRUTA + arquivo, arquivo)
        caminhos[nome] = arquivo

for nome, caminho in caminhos.items():
    print("%-16s -> %s" % (nome, caminho))


## 2. Carregar as cinco séries em `DataFrame`

Um dicionário de `DataFrame`, um por série. `periodo` e `unidade` chegam como texto
(`dtype=object`); `valor` chega como número (`float64`). Nenhuma conversão para data: `"2025-T4"`
não é uma data de calendário, é um rótulo de trimestre, e forçar um mês arbitrário reintroduziria o
erro de granularidade que a Aula 02 evitou.


In [ ]:
series = {nome: pd.read_csv(caminho) for nome, caminho in caminhos.items()}

for nome, df in series.items():
    print("%-16s %3d linhas  colunas=%s" % (nome, len(df), list(df.columns)))

print()
print(series["abate_bovinos"].dtypes)


## 3. `describe()`

Sete estatísticas de uma vez: `count`, `mean`, `std`, `min`, `25%`, `50%` (mediana), `75%`, `max`.


In [ ]:
series["producao_ovos"].describe()


`count` aparece como **157**, não 117: é a mesma diferença de dez anos que a Aula 02 mediu na mão
(ovos começa em `1987-T1`; as outras quatro séries, em `1997-T1`). Confira o `count` das outras
quatro na célula abaixo.


In [ ]:
for nome, df in series.items():
    print("%-16s count=%d" % (nome, df["valor"].count()))


## 4. Estatística descritiva detalhada

Média, mediana, desvio padrão e coeficiente de variação (desvio padrão dividido pela média, em
percentual) das cinco séries. Compare a faixa de mínimo e máximo com a seção "Checagem de sanidade
dos valores" de `dados/README.md`: as cinco precisam bater.


In [ ]:
print("%-16s %8s %10s %10s %10s %8s" % ("serie", "count", "media", "mediana", "desvio", "cv%"))
for nome, df in series.items():
    v = df["valor"]
    cv = v.std() / v.mean() * 100
    print("%-16s %8d %10.3e %10.3e %10.3e %7.1f%%"
          % (nome, v.count(), v.mean(), v.median(), v.std(), cv))


Repare que `producao_ovos` e `abate_suinos` têm o **maior** coeficiente de variação (perto de 44%),
maior até que `producao_leite`, que no bloco seguinte vai ter o salto trimestral mais violento das
cinco. Não é contradição: o coeficiente de variação, calculado sobre décadas de série, mistura
**crescimento de longo prazo** com **oscilação trimestre a trimestre**. Separar essas duas coisas é
o trabalho da Aula 04.


## 5. Numpy: variação percentual trimestre a trimestre

`np.diff(valores)` devolve um array com um elemento a menos que `valores`: a posição `0` do
resultado é `valores[1] - valores[0]`. Por isso o denominador da variação percentual usa
`valores[:-1]` (todo mundo, menos o último): cada diferença é comparada com o valor **anterior**,
nunca com o seguinte.


In [ ]:
def maior_salto(df):
    valores = df["valor"].to_numpy()
    periodos = df["periodo"].to_numpy()
    variacao = (np.diff(valores) / valores[:-1]) * 100
    idx = np.argmax(np.abs(variacao))
    media_abs = np.mean(np.abs(variacao))
    return periodos[idx], periodos[idx + 1], variacao[idx], media_abs, variacao

print("%-16s %10s %10s %10s %14s" % ("serie", "de", "para", "salto%", "media|var|%"))
variacoes = {}
for nome, df in series.items():
    de, para, salto, media, variacao = maior_salto(df)
    variacoes[nome] = variacao
    print("%-16s %10s %10s %9.2f%% %13.2f%%" % (nome, de, para, salto, media))


`producao_leite` tem o maior salto (perto de +19%) e também a maior variação média em módulo: é a
série mais instável trimestre a trimestre, o oposto do que o coeficiente de variação da seção 4
sugeria isoladamente. As duas métricas respondem perguntas diferentes.


## 6. Visualização: sazonalidade e outliers

Dois gráficos, com dado real. O primeiro plota `abate_bovinos` no tempo e marca o maior salto
encontrado na seção 5. O segundo agrupa `producao_leite` por trimestre do ano e mostra a média de
cada grupo, procurando sazonalidade.


In [ ]:
df = series["abate_bovinos"]
valores = df["valor"].to_numpy()
periodos = df["periodo"].to_numpy()
variacao = (np.diff(valores) / valores[:-1]) * 100
idx_salto = np.argmax(np.abs(variacao)) + 1  # +1: o salto e do periodo anterior PARA este indice

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(range(len(df)), valores / 1e9, color="#2e2640", linewidth=1.4)
ax.scatter([idx_salto], [valores[idx_salto] / 1e9], color="#ff4545", zorder=5)
ticks = range(0, len(df), 16)
ax.set_xticks(list(ticks))
ax.set_xticklabels(periodos[list(ticks)])
ax.set_xlabel("periodo (trimestre)")
ax.set_ylabel("bilhoes de kg")
ax.set_title("Abate de bovinos: o maior salto (%s -> %s) marcado em vermelho"
             % (periodos[idx_salto - 1], periodos[idx_salto]))
fig.tight_layout()
plt.show()


In [ ]:
leite = series["producao_leite"].copy()
leite["trimestre_do_ano"] = leite["periodo"].str.split("-T").str[1].astype(int)
media_por_trimestre = leite.groupby("trimestre_do_ano")["valor"].mean() / 1e6

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.barplot(x=media_por_trimestre.index, y=media_por_trimestre.values,
            color="#2e2640", ax=ax)
ax.set_xlabel("trimestre do ano")
ax.set_ylabel("milhoes de mil litros (media)")
ax.set_title("Leite adquirido: media por trimestre do ano")
fig.tight_layout()
plt.show()

media_por_trimestre


O segundo gráfico mostra **que** existe um padrão sazonal estável (T2 sempre o mais baixo, T4 o
mais alto); ele não mostra **por que**. Nenhum dos cinco CSVs tem coluna de clima ou de estação do
ano: qualquer explicação (por exemplo, pastagem e período de seca) é hipótese, não conclusão, o
mesmo cuidado que a Aula 02 pediu para análise diagnóstica.


## 7. Dados faltantes

`isna()` é o primeiro comando de qualquer projeto novo. Rode nas cinco séries.


In [ ]:
for nome, df in series.items():
    vazios = df["valor"].isna().sum()
    print("%-16s %3d vazios de %3d" % (nome, vazios, len(df)))
    assert vazios == 0, "%s tem valor ausente" % nome

print()
print("as cinco series tem zero valores ausentes")


Zero não é acaso: `tools/baixar_dados.py` já filtra os marcadores de ausência do SIDRA
(`"..."`, `"-"`, `"X"`, `"*"`) antes de gravar o CSV. O que `isna()` confirma aqui é que o filtro
está funcionando, não que a base não tem risco de qualidade: a Aula 02 mostrou que
`producao_leite` passa em `isna()` e ainda assim mede o volume **adquirido** pelos laticínios, não
o **produzido**. Zero vazios não é zero risco.


## 8. Desafio

Responda no código, e o item 3 em texto.

1. Calcule o coeficiente de variação (`std / mean * 100`) das cinco séries e ordene da maior para a
   menor.
2. Agrupando `producao_leite` por trimestre do ano, qual trimestre (1 a 4) tem a **maior** média
   histórica?
3. Escolham a série que a dupla explorou nos exercícios de hoje. Em `resposta_3`, escrevam uma
   frase sobre um padrão (sazonalidade ou outlier) que vocês notaram nela, e uma hipótese não
   confirmável só com este dado sobre o que poderia explicar esse padrão.


In [ ]:
# 1. coeficiente de variacao, do maior para o menor
cvs = {nome: df["valor"].std() / df["valor"].mean() * 100 for nome, df in series.items()}
for nome, cv in sorted(cvs.items(), key=lambda par: par[1], reverse=True):
    print("%-16s %5.1f%%" % (nome, cv))

# 2. trimestre do ano com a maior media historica de leite
leite = series["producao_leite"].copy()
leite["trimestre_do_ano"] = leite["periodo"].str.split("-T").str[1].astype(int)
media_por_trimestre = leite.groupby("trimestre_do_ano")["valor"].mean()
tri_pico = media_por_trimestre.idxmax()
print()
print("trimestre de maior media historica de leite: T%d" % tri_pico)

# 3. escreva a sua resposta aqui, em uma ou duas frases
resposta_3 = ""
print()
print("resposta_3:", repr(resposta_3))
